## Vectorization
### 1. The Nature of Vectorization
Vectorization is essentially the programming art of removing explicit for-loops from your source code.

In the modern Deep Learning era, models are typically trained on massive datasets (millions of images or text snippets). If you use a traditional sequential loop structure to process each data point one by one, the algorithm will run incredibly slowly, wasting resources and time. The skill of applying Vectorization is mandatory for deploying AI models in practice.

<img src="../images/Week02/image15.png" width=800>

### 2. Analyzing the Computational Problem: Logistic Regression
To illustrate, let us consider the equation for calculating the linear value $z$ in the Logistic Regression model:
$$
z = w^T x + b
$$
Here, $w$ and $x$ are two column vectors with the same dimensionality (for example, 1 million dimensions). To multiply these two vectors together, we have two approaches when programming in Python.

**Approach 1:** Non-vectorized implementation (Using a for loop)
You would have to initialize $z = 0$, then iterate through the 1 million elements of the two vectors, multiply each pair of numbers, accumulate the sum, and finally add the bias $b$.

In [1]:
import numpy as np
import time

w = np.random.rand(1000000)
x = np.random.rand(1000000)
b = np.random.rand(1)
z = 0.0
start_time = time.time()
for i in range(1000000):
    z += w[i] * x[i]
z += b
end_time = time.time()
print("Time taken: {:.6f} seconds".format(end_time - start_time))

Time taken: 0.383973 seconds


**Approach 2:** Vectorized implementation (Using the NumPy library)
You simply call a single function to perform direct matrix multiplication, then directly add the bias $b$.

In [2]:
start_time = time.time()
z = np.dot(w, x) + b
end_time = time.time()
print("Time taken with NumPy: {:.6f} seconds".format(end_time - start_time))

Time taken with NumPy: 0.004909 seconds


### 3. Performance Proof
The results show an astonishing performance gap:
- **Non-vectorized implementation (using a for loop):** Completing the calculation line-by-line takes 0.383973 seconds (approximately 384 milliseconds).
- **Vectorized implementation (using the NumPy library):** Completing the exact same calculation takes only 0.004909 seconds (approximately 4.9 milliseconds).

By merely replacing the explicit loop with a vectorized function, the algorithm runs roughly **78 times faster**. When training Deep Learning models on massive datasets, operations like this are repeated millions of times. This 78x optimization factor is the critical boundary between a model that takes 1 minute to train and one that forces you to wait over an hour for the exact same result.

### 4. The Secret Behind the Power of Vectorization
Why does removing a for loop make the code run so fast? The answer lies in computer hardware architecture, specifically **SIMD (Single Instruction Multiple Data)**.
- When you use a `for` loop, the CPU (or GPU) operates sequentially: It reads the multiplication instruction, takes the 1st number of $w$ and multiplies it by the 1st number of $x$, and stores the result. Then it repeats the process for the 2nd number... Doing this 1 million times.
- When you use vectorized functions like `np.dot()`, the NumPy library (which is highly optimized in C/C++) takes advantage of SIMD instructions. SIMD allows the hardware to grab a very large chunk of data at once (for instance, 100 numbers from $w$ and 100 numbers from $x$) and apply the multiplication operation to all those pairs in a single clock cycle, running them in parallel.

Although GPUs (Graphics Processing Units) are famously known as the kings of SIMD parallel computing, your computer's CPU is also highly capable of executing SIMD instructions powerfully if the source code is written correctly (vectorized).

When programming in Deep Learning, try your absolute best to avoid using explicit for loops. Whenever possible, call built-in NumPy functions to process matrices.

## Vectorizing Logistic Regression
### 1. The Problem with the Loop Method
If you do it manually, to predict results for $m$ training examples, you would have to run a loop from the 1st to the $m$-th example. At each iteration $i$, you have to compute two equations:
- Calculate the linear value: $z^{(i)} = w^T x^{(i)} + b$
- Calculate the activation (predicted) value: $a^{(i)} = \sigma(z^{(i)})$

Calculating this repeatedly $m$ times is incredibly slow. Our goal is to group all $m$ of these calculations and solve them simultaneously.

### 2. Vectorizing the Linear Value Calculation
Instead of processing each column vector $x^{(i)}$ individually, we stack all $m$ of these vectors together to form a massive input matrix $X$ with the shape $(n_x, m)$.

Similarly, instead of calculating individual values $z^{(1)}, z^{(2)}, \dots, z^{(m)}$, we will stack all of them into a single row vector with the shape $(1, m)$, called matrix $Z$.

Matrix mathematics allows us to compute this entire matrix $Z$ through a single equation:
$$
Z = w^T X + [b, b, \dots, b]
$$

Let's break down this multiplication:
- $w^T$ is a row vector with the shape $(1, n_x)$.
- $X$ is the input matrix with the shape $(n_x, m)$.
- When multiplying the matrix $w^T$ by the matrix $X$, according to the rules of linear algebra, the result will automatically be a new matrix with the shape $(1, m)$. Each element in this new matrix corresponds exactly to the $w^T x^{(i)}$ term of each data sample.

**The Concept of Broadcasting in Python:**

In the mathematical equation above, you see the addition of a row vector consisting entirely of $b$ coefficients $[b, b, \dots, b]$. However, when programming with the NumPy library, you simply need to directly add the single real number $b$ to the resulting matrix. Python will infer your intention and automatically duplicate (copy) that number $b$ into a row vector of length $m$ to add it to each corresponding element. This automatic variable resizing mechanism is called **Broadcasting**.

Thanks to this, the entire loop to calculate $z$ for $m$ elements is reduced to a single line of Python code:

```python
Z = np.dot(w.T, X) + b
```

### 3. Vectorizing the Prediction Calculation
Once you have the matrix $Z$ containing all the linear values, the next step is to calculate the predicted (activation) value $a^{(i)}$ for each sample.

Similar to how $X$ and $Z$ were defined, we group all the results $a^{(1)}, a^{(2)}, \dots, a^{(m)}$ into a row vector of shape $(1, m)$, called matrix $A$.

To compute $A$, we simply apply the Sigmoid function to the entire matrix $Z$ simultaneously. The NumPy library allows you to write a Sigmoid function that can accept a matrix as input, and it will automatically compute the Sigmoid for each individual element inside that matrix independently and in parallel.
$$
A = \sigma(Z)
$$

### 4. Summary
With just the support of Linear Algebra and the NumPy library, the entire forward propagation process to calculate predictions for millions of data points has been compressed into 2 extremely neat lines of code that run at lightning speed:

```python
Z = np.dot(w.T, X) + b
A = sigmoid(Z)
```

<img src="../images/Week02/image16.png" width=800>

## Vectorizing Logistic Regression's Gradient Output
### 1. Vectorizing the Error
During backpropagation, the first step is to compute the error $dz^{(i)}$ for each example. According to the original formula, we have $dz^{(i)} = a^{(i)} - y^{(i)}$.

Instead of using a loop to calculate each value individually, we group them into a matrix. As previously computed, the prediction matrix is $A = [a^{(1)}, a^{(2)}, \dots, a^{(m)}]$. The actual label matrix is $Y = [y^{(1)}, y^{(2)}, \dots, y^{(m)}]$.

Since both matrices have the dimension $(1, m)$, direct matrix subtraction can be performed. The result is a new row vector $dZ$ containing the errors for the entire dataset:
$$
dZ = A - Y
$$

### 2. Vectorizing the Parameter Derivatives
This is the core step to completely eliminate the loop over the $m$ examples.

**Computing the derivative of the bias (db):**

According to the original algorithm, $db$ is the average of all $dz^{(i)}$ values. Therefore, we apply the NumPy sum function to the matrix $dZ$, then divide by $m$:
$$
db = \frac{1}{m} \text{np.sum}(dZ)
$$

**Computing the derivative of the weights (dw):**

According to the original algorithm, for each $j$-th feature, the product of $x_j^{(i)}$ and $dz^{(i)}$ is summed across the $m$ examples. Mathematically, this process is equivalent to the matrix multiplication of the input data matrix $X$ and the transpose of the error matrix $dZ^T$.

Analyzing the matrix multiplication $X \times dZ^T$:
- Matrix $X$ has the dimension $(n_x, m)$, containing all input features.
- Matrix $dZ^T$ is a column vector with the dimension $(m, 1)$, containing all the errors.
- Following the rule of matrix multiplication (row by column), the first element of the result is computed by multiplying the first row of $X$ (containing feature $x_1$ across all examples) by the column of $dZ^T$ (containing the errors of all examples) and summing the products.

The final result is a column vector $dw$ of dimension $(n_x, 1)$, which matches the original dimension of the weight vector $w$.

This formula is implemented in Python as follows:
$$
dw = \frac{1}{m} X \cdot dZ^T
$$

### 3. Complete Overview
A complete step (one epoch) of Gradient Descent can now be written without using any explicit for loops over the $m$ examples or the $n_x$ features:
```python
# 1. Forward Propagation (Computing predictions)
Z = np.dot(w.T, X) + b
A = sigmoid(Z)

# 2. Backward Propagation (Computing errors and derivatives)
dZ = A - Y
dw = (1/m) * np.dot(X, dZ.T)
db = (1/m) * np.sum(dZ)

# 3. Parameter Update
w = w - alpha * dw
b = b - alpha * db
```
**Important Note:** While the data processing loops have been successfully vectorized, the Gradient Descent algorithm is inherently an iterative process to reach convergence. Therefore, an outer for loop is still required to repeat the three-step block above (e.g., a loop running for 1000 epochs). Vectorization accelerates each of these iterations.